# CDFI DiD Estimation Pipeline

Callaway & Sant'Anna (2021) Difference-in-Differences with Neural Network Nuisance Estimation

---

## Cell 1: Setup - Paths and Dependencies

In [ ]:
# Set your project root (adjust for your environment)
import sys
import os

project_root = "/path/to/your/project"  # <-- CHANGE THIS
os.chdir(project_root)

# Add modules to path
sys.path.insert(0, os.path.join(project_root, "code/python"))

# Install packages if needed (uncomment if first run)
# !pip install torch pandas numpy matplotlib scipy

# Import modules
from modules import (
    create_config, set_seed, print_config, get_device,
    load_panel_data, get_gt_pairs, summarize_data,
    get_covariate_info, create_covariate_masks, validate_covariate_rules,
    create_model,
    run_cross_fitting, validate_cross_fitting,
    diagnose_nuisance, compute_all_att, print_att_summary,
    add_bootstrap_inference, test_parallel_trends, compute_simple_att,
    aggregate_all, print_aggregation_summary,
    plot_event_study, save_event_study, generate_all_figures,
    Timer
)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print("\nSetup complete.")

## Cell 2: Configuration

In [ ]:
# Create configuration
config = create_config(
    # Outcome variable (without y_ prefix)
    outcome="sfr_pc",
    
    # Neural network architecture
    architecture={
        'input_projection_dim': 128,
        'shared_layers': [256, 128],
        'outcome_head_layers': [64],
        'propensity_head_layers': [64],
        'activation': 'relu',
        'dropout': 0.1,
        'layer_norm': True
    },
    
    # Training settings
    training={
        'epochs': 100,
        'batch_size': 256,
        'validation_split': 0.2,
        'early_stopping': {
            'enabled': True,
            'patience': 15,
            'min_delta': 1e-4
        }
    },
    
    # Cross-fitting
    cross_fitting={
        'n_folds': 2,
        'seed': 42
    },
    
    # Inference
    inference={
        'n_bootstrap': 1000,
        'alpha': 0.05
    },
    
    # Monitoring
    monitoring={
        'verbose': True,
        'print_every': 5
    }
)

# Print configuration
print_config(config)

# Set random seed
set_seed(config.seed)

# Check device
device = get_device(config)
print(f"Using device: {device}")

## Cell 3: Load Data

In [ ]:
data_path = os.path.join(project_root, "data/analysis/final_analysis_dataset.csv")

# Load data (set sample_n for testing, None for full dataset)
sample_n = None  # e.g., 50000 for testing

print("Loading data...")
timer = Timer()

data, metadata = load_panel_data(data_path, config, sample_n=sample_n)

print(f"\nData loaded in {timer.elapsed():.1f} seconds")

# Print summary
summarize_data(data, metadata, config)

## Cell 4: Setup (g,t) Pairs and Covariates

In [ ]:
print("Setting up (g,t) pairs and covariates...")
timer = Timer()

# Get valid (g,t) pairs
gt_pairs = get_gt_pairs(data, config)

# Get covariate information
covariate_info = get_covariate_info(gt_pairs, data.columns.tolist(), config)

# Create covariate masks (for validation)
covariate_masks = create_covariate_masks(gt_pairs, covariate_info.all_covariates, config)

# Validate
validate_covariate_rules(config)

print(f"\nSetup complete in {timer.elapsed():.1f} seconds")
print(f"\nSummary:")
print(f"  (g,t) pairs: {len(gt_pairs)} ({gt_pairs['is_pre'].sum()} pre, {(~gt_pairs['is_pre']).sum()} post)")
print(f"  Covariates: {len(covariate_info.all_covariates)} (dims per (g,t): {min(covariate_info.dims_by_gt.values())}-{max(covariate_info.dims_by_gt.values())})")

## Cell 5: Cross-Fitting (Main Training)

**This is the computationally intensive step.**

In [ ]:
print("="*60)
print("CROSS-FITTING")
print("="*60 + "\n")

timer = Timer()

cf_results = run_cross_fitting(
    data,
    gt_pairs,
    covariate_info.all_covariates,
    covariate_masks,
    config
)

training_time = timer.elapsed()
print(f"\nCross-fitting complete in {training_time:.1f} seconds ({training_time/60:.1f} minutes)")

# Validate
validate_cross_fitting(cf_results, config)

## Cell 6: ATT Estimation

In [ ]:
print("="*60)
print("ATT ESTIMATION")
print("="*60 + "\n")

timer = Timer()

# Diagnose nuisance parameters
nuisance_diag = diagnose_nuisance(cf_results, config)

# Compute all ATT(g,t) estimates
att_results = compute_all_att(cf_results, config)

# Print summary
print_att_summary(att_results)

print(f"\nATT estimation complete in {timer.elapsed():.1f} seconds")

## Cell 7: Inference (Bootstrap)

In [ ]:
print("="*60)
print("INFERENCE")
print("="*60 + "\n")

timer = Timer()

# Add bootstrap inference
att_results = add_bootstrap_inference(att_results, config)

# Test parallel trends
pt_test = test_parallel_trends(att_results, config)

# Compute simple ATT
simple_att = compute_simple_att(att_results, config)

print(f"\nInference complete in {timer.elapsed():.1f} seconds")

# Print results
print("\n--- Parallel Trends Test ---")
print(f"Mean pre-treatment ATT: {pt_test['mean_att_pre']:.4f} (SE: {pt_test['se_mean_pre']:.4f})")
print(f"Test statistic: {pt_test['test_stat']:.3f}, p-value: {pt_test['p_value']:.4f}")
print(f"Reject parallel trends at 5%: {'YES' if pt_test['reject'] else 'NO'}")

print("\n--- Simple ATT (Weighted Average) ---")
print(f"ATT: {simple_att['att']:.4f} (SE: {simple_att['se']:.4f})")
print(f"95% CI: [{simple_att['ci_lower']:.4f}, {simple_att['ci_upper']:.4f}]")
print(f"p-value: {simple_att['p_value']:.4f}")

## Cell 8: Aggregation

In [ ]:
print("="*60)
print("AGGREGATION")
print("="*60 + "\n")

agg_results = aggregate_all(att_results, config)

# Print summary
print_aggregation_summary(agg_results)

# View event study table
print("\n--- Event Study Estimates ---")
es_df = agg_results['event_study']['event_study']
print(es_df[['event_time', 'att', 'se', 'ci_lower', 'ci_upper']].to_string(index=False))

## Cell 9: Visualization

In [ ]:
print("="*60)
print("VISUALIZATION")
print("="*60 + "\n")

# Create output directory
output_dir = os.path.join(project_root, "outputs/figures")
os.makedirs(output_dir, exist_ok=True)

# Generate event study plot
es_fig = plot_event_study(
    agg_results['event_study'],
    title="Effect of CDFI Lending on Startup Formation Rate",
    subtitle="Callaway & Sant'Anna (2021) DiD with Neural Network Nuisance Estimation",
    show_uniform_bands=True,
    show_pointwise_ci=True
)

# Display plot
plt.show()

# Save plots
save_event_study(es_fig, "event_study", output_dir)

print(f"\nPlots saved to: {output_dir}")

## Cell 10: Save Results

In [ ]:
import pickle

# Compile all results
results = {
    'config': config,
    'metadata': metadata,
    'gt_pairs': gt_pairs,
    'covariate_info': covariate_info,
    'att_results': att_results,
    'agg_results': agg_results,
    'parallel_trends_test': pt_test,
    'simple_att': simple_att,
    'nuisance_diagnostics': nuisance_diag,
    'training_time': training_time
}

# Save as pickle
output_path = os.path.join(project_root, "outputs/estimation_results.pkl")
with open(output_path, 'wb') as f:
    pickle.dump(results, f)

print(f"Results saved to: {output_path}")

## Cell 11: Results Summary

In [ ]:
print("\n" + "="*60)
print("ESTIMATION COMPLETE")
print("="*60 + "\n")

print("KEY RESULTS:\n")

print("1. Simple ATT (weighted average post-treatment):")
print(f"   ATT = {simple_att['att']:.4f}, SE = {simple_att['se']:.4f}, p = {simple_att['p_value']:.4f}")
print(f"   95% CI: [{simple_att['ci_lower']:.4f}, {simple_att['ci_upper']:.4f}]\n")

print("2. Parallel Trends:")
print(f"   Pre-treatment ATT = {pt_test['mean_att_pre']:.4f} (should be ~0)")
print(f"   p-value = {pt_test['p_value']:.4f} (want > 0.05)\n")

print("3. Event Study:")
es = agg_results['event_study']['event_study']
print(f"   Event times: {es['event_time'].min()} to {es['event_time'].max()}")
print(f"   Pre-treatment mean: {es[es['event_time'] < 0]['att'].mean():.4f}")
print(f"   Post-treatment mean: {es[es['event_time'] >= 0]['att'].mean():.4f}")

print(f"\nTraining time: {training_time/60:.1f} minutes")
print(f"Figures saved to: {output_dir}")
print(f"Results saved to: {output_path}")